In [ ]:
import sys
import os
# Add the parent directory (root of the project) to the path
sys.path.append(os.path.abspath('..'))

import shared


import polars as pl


import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#runner_name = "Anniina Silvennoinen"
#runner2_name = "Harry Jokela"
race_type, names = "ve", ["Pauliina Mäkelä", "Pauliina Åkerlund"]
race_type, names = "ju", ["Saku Laine"]
race_type, names = "ju", ["Milja Mäkinen", "Milja Kallio"]

race_type, names = "ve", ["Piia Ruuskanen", "Piia Heiniö"]
race_type, names = "ju", ["Tuomas Soisalo"]


names = [name.lower() for name in names]



os.environ['RACE_TYPE'] = race_type
os.environ['FORECAST_YEAR'] = "2026"


names

In [ ]:
runs_path = f"../data/long_runs_and_running_order_{shared.race_id_str()}.tsv"

runs_df = pl.read_csv(runs_path, separator="\t")
runs_df

In [ ]:
pl.Config.set_tbl_rows(40)
names_pattern = '|'.join(names)
runs_df.filter(pl.col('name').str.to_lowercase().str.contains(names_pattern)).select([
 'run_id',
 'unique_name',
 'name',
 'team_id',
 'team',
 'team_country',
 'year',
 'pace',
 'emit',
 'leg',
 'ro_orig_name',
 'run_num',
 'median_pace',
 'num_runs',])

In [ ]:
runs_df = runs_df.with_columns(
    z_score=((pl.col("pace") - pl.col("median_pace")) / pl.col("log_stdev").exp())
)

In [ ]:
runs_df.filter(pl.col('name').str.to_lowercase().str.contains(names_pattern))

In [ ]:
runs_df.filter(pl.col('name').str.to_lowercase().str.contains(names_pattern)).select([
 'year',
 'team_id',
 'team',
 'leg',
 'name',
 'emit',
 'run_num',
 'pace',
 'median_pace',])

In [ ]:
filtered_runs = runs_df.filter(pl.col('name').str.to_lowercase().str.contains(names_pattern))
sns.scatterplot(data=filtered_runs.to_pandas() if hasattr(filtered_runs, 'to_pandas') else filtered_runs, x="year", y="pace", hue="unique_name")
plt.axhline(y=7.873, color='y', linestyle='--', zorder=-1)

In [ ]:
sns.lmplot(data=filtered_runs.to_pandas(), x="year", y="pace", hue="unique_name")

In [ ]:

estimates = pl.read_csv(f'../data/running_order_with_estimates_{shared.race_id_str()}.tsv', separator="\t")

In [ ]:
estimates = (
    estimates.filter(pl.col('name').is_not_null())
    .with_columns(
        final_pace=pl.col('log_mean').exp(),
        final_std=pl.col('log_std').exp()
    )
)
estimates.filter(pl.col('name').str.to_lowercase().str.contains(names_pattern))
# estimates.filter(pl.col('name').is_null())
